In [13]:
!pip install transformers textblob textstat nltk
!pip install datasets pandas

In [14]:
import pandas as pd
from datasets import load_dataset
import kagglehub
import os
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from transformers import RobertaModel, RobertaTokenizer
from textblob import TextBlob
import textstat
import nltk

nltk.download('punkt')

[nltk_data] Downloading package punkt to /root/nltk_data...
[nltk_data]   Package punkt is already up-to-date!


True

In [15]:
import torch
import torch.nn as nn
from torch.optim import AdamW
from tqdm import tqdm
import time
from sklearn.metrics import classification_report, accuracy_score
from transformers import get_linear_schedule_with_warmup

### The Dual-Stream PyTorch Implementation

In [16]:
# ==========================================
# 1. Linguistic Feature Extraction Pipeline
# ==========================================
def extract_linguistic_features(text):
    """Extracts explicit psycholinguistic and statistical features."""
    if not isinstance(text, str) or len(text.strip()) == 0:
        return [0.0, 0.0, 0.0, 0.0]

    # 1. Sentiment and Subjectivity
    blob = TextBlob(text)
    polarity = blob.sentiment.polarity
    subjectivity = blob.sentiment.subjectivity

    # 2. Readability Index (Flesch-Kincaid Grade)
    readability = textstat.flesch_kincaid_grade(text)

    # 3. Lexical Diversity
    words = text.split()
    lexical_diversity = len(set(words)) / len(words) if len(words) > 0 else 0.0

    return [polarity, subjectivity, readability, lexical_diversity]

# ==========================================
# 2. Custom Dataset Class
# ==========================================
class FakeNewsDataset(Dataset):
    def __init__(self, texts, labels, tokenizer, max_len=256):
        self.texts = texts
        self.labels = labels
        self.tokenizer = tokenizer
        self.max_len = max_len

    def __len__(self):
        return len(self.texts)

    def __getitem__(self, idx):
        text = str(self.texts[idx])
        label = self.labels[idx]

        # Contextual Stream (Transformer Encoding)
        encoding = self.tokenizer(
            text,
            add_special_tokens=True,
            max_length=self.max_len,
            padding='max_length',
            truncation=True,
            return_tensors='pt'
        )

        # Linguistic Feature Stream
        ling_features = extract_linguistic_features(text)

        return {
            'input_ids': encoding['input_ids'].flatten(),
            'attention_mask': encoding['attention_mask'].flatten(),
            'ling_features': torch.tensor(ling_features, dtype=torch.float),
            'labels': torch.tensor(label, dtype=torch.long)
        }

# ==========================================
# 3. Dual-Stream Feature Fusion Network
# ==========================================
class DualStreamFakeNewsClassifier(nn.Module):
    def __init__(self, model_name='roberta-base', num_ling_features=4, num_classes=2):
        super(DualStreamFakeNewsClassifier, self).__init__()

        # Stream 1: Pre-trained Transformer Encoder
        self.roberta = RobertaModel.from_pretrained(model_name)
        hidden_size = self.roberta.config.hidden_size

        # Stream 2 is passed directly as an input tensor in the forward pass.
        self.layer_norm = nn.LayerNorm(hidden_size + num_ling_features)
        # Feature Fusion & Classification Head
        # Concatenated size = RoBERTa hidden size (768) + 4 linguistic features
        self.classifier = nn.Sequential(
            nn.Linear(hidden_size + num_ling_features, 512),
            nn.ReLU(),
            nn.Dropout(0.3),
            nn.Linear(512, 128),
            nn.ReLU(),
            nn.Dropout(0.2),
            nn.Linear(128, num_classes)
        )

    def forward(self, input_ids, attention_mask, ling_features):
        # Extract RoBERTa [CLS] token representation
        outputs = self.roberta(input_ids=input_ids, attention_mask=attention_mask)
        cls_embedding = outputs.last_hidden_state[:, 0, :] # Shape: (batch_size, 768)

        # Fuse explicit features with implicit context embeddings
        fused_vector = torch.cat((cls_embedding, ling_features), dim=1) # Shape: (batch_size, 772)
        fused_vector = self.layer_norm(fused_vector)
        # Pass through MLP to get final logits
        logits = self.classifier(fused_vector)
        return logits

print("Dual-Stream Architecture Successfully Initialized.")

Dual-Stream Architecture Successfully Initialized.


### Importing and Processing Datasets

*   FakeNewsNet - https://github.com/KaiDMML/FakeNewsNet/tree/master/dataset
*   LIAR - https://www.kaggle.com/datasets/doanquanvietnamca/liar-dataset

In [17]:
print("Loading Tokenizer...")
tokenizer = RobertaTokenizer.from_pretrained('roberta-base')

# ==========================================
# 1. Import and Preprocess LIAR Dataset (via Kaggle)
# ==========================================
print("\nDownloading LIAR Dataset from Kaggle...")
path = kagglehub.dataset_download("doanquanvietnamca/liar-dataset")

# The TSV files don't have headers, so we define them based on the README
liar_columns = [
    'id', 'label', 'statement', 'subject', 'speaker', 'job', 'state', 'party',
    'barely_true_counts', 'false_counts', 'half_true_counts', 'mostly_true_counts', 'pants_fire_counts', 'context'
]

# Load Train and Validation sets
liar_train_df = pd.read_csv(os.path.join(path, 'train.tsv'), sep='\t', names=liar_columns)
liar_val_df = pd.read_csv(os.path.join(path, 'valid.tsv'), sep='\t', names=liar_columns)

# Binarize labels: 'pants-fire', 'false', 'barely-true' -> 1 (Fake)
# 'half-true', 'mostly-true', 'true' -> 0 (Real)
fake_labels = ['pants-fire', 'false', 'barely-true']
liar_train_df['binary_label'] = liar_train_df['label'].apply(lambda x: 1 if x in fake_labels else 0)
liar_val_df['binary_label'] = liar_val_df['label'].apply(lambda x: 1 if x in fake_labels else 0)

# Extract statements and labels
liar_train_texts = liar_train_df['statement'].fillna("").tolist()
liar_train_labels = liar_train_df['binary_label'].tolist()

liar_val_texts = liar_val_df['statement'].fillna("").tolist()
liar_val_labels = liar_val_df['binary_label'].tolist()

print(f"LIAR loaded successfully: {len(liar_train_texts)} training samples, {len(liar_val_texts)} validation samples.")

# ==========================================
# 2. Import and Preprocess FakeNewsNet (Local CSVs)
# ==========================================
print("\nPreparing FakeNewsNet...")

# Dictionary mapping your expected filenames to their true/fake binary labels
fnn_files = {
    'gossipcop_real.csv': 0,
    'gossipcop_fake.csv': 1,
    'politifact_real.csv': 0,
    'politifact_fake.csv': 1
}

fnn_texts = []
fnn_labels = []

# Loop through the files, extract the 'title' column, and assign the label
for filename, label in fnn_files.items():
    if os.path.exists(filename):
        df = pd.read_csv(filename)
        # Extract titles as the text feature
        titles = df['title'].fillna("").tolist()
        fnn_texts.extend(titles)
        fnn_labels.extend([label] * len(titles))
        print(f" -> Loaded {len(titles)} samples from {filename}")
    else:
        print(f" -> WARNING: {filename} not found in current directory. Please upload it.")

if len(fnn_texts) > 0:
    print(f"FakeNewsNet loaded successfully: {len(fnn_texts)} total samples.")
else:
    print("WARNING: No FakeNewsNet data loaded. Proceeding with LIAR data only for now.")

# ==========================================
# 3. Instantiate the PyTorch Datasets
# ==========================================
print("\nWrapping datasets into PyTorch DataLoader...")

# Using the FakeNewsDataset class we defined previously
liar_pytorch_train = FakeNewsDataset(liar_train_texts, liar_train_labels, tokenizer)
liar_pytorch_val = FakeNewsDataset(liar_val_texts, liar_val_labels, tokenizer)

# Create the DataLoaders (Batching)
train_loader = DataLoader(liar_pytorch_train, batch_size=32, shuffle=True)
val_loader = DataLoader(liar_pytorch_val, batch_size=32, shuffle=False)

print("Data pipeline complete! Ready to begin training loops.")

Loading Tokenizer...

Using Colab cache for faster access to the 'liar-dataset' dataset.
LIAR loaded successfully: 10240 training samples, 1284 validation samples.

Preparing FakeNewsNet...
 -> Loaded 16817 samples from gossipcop_real.csv
 -> Loaded 5323 samples from gossipcop_fake.csv
 -> Loaded 624 samples from politifact_real.csv
 -> Loaded 432 samples from politifact_fake.csv
FakeNewsNet loaded successfully: 23196 total samples.

Wrapping datasets into PyTorch DataLoader...
Data pipeline complete! Ready to begin training loops.


In [18]:
# ==========================================
# 1. Initialization and Setup
# ==========================================
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Training on device: {device}")

# Instantiate the model
model = DualStreamFakeNewsClassifier(model_name='roberta-base', num_ling_features=4, num_classes=2)
model = model.to(device)

# Differential Learning Rates Setup
optimizer_grouped_parameters = [
    {'params': model.roberta.parameters(), 'lr': 2e-5},      # Gentle fine-tuning for RoBERTa
    {'params': model.classifier.parameters(), 'lr': 1e-3}    # Aggressive learning for our new MLP
]

optimizer = AdamW(optimizer_grouped_parameters)
criterion = nn.CrossEntropyLoss()

epochs = 3
total_steps = len(train_loader) * epochs

# Set up the scheduler
scheduler = get_linear_schedule_with_warmup(
    optimizer,
    num_warmup_steps=int(0.1 * total_steps), # 10% of training is warmup
    num_training_steps=total_steps
)

# ==========================================
# 2. The Training Loop
# ==========================================
print("\nStarting Training Phase...")

for epoch in range(epochs):
    print(f"\n======== Epoch {epoch + 1} / {epochs} ========")
    start_time = time.time()

    # --- TRAINING ---
    model.train()
    total_train_loss = 0

    # tqdm gives us a nice progress bar in Colab
    train_loop = tqdm(train_loader, leave=True, position=0)
    for batch in train_loop:
        # Move tensors to GPU
        input_ids = batch['input_ids'].to(device)
        attention_mask = batch['attention_mask'].to(device)
        ling_features = batch['ling_features'].to(device)
        labels = batch['labels'].to(device)

        # Clear previous gradients
        optimizer.zero_grad()

        # Forward pass
        logits = model(input_ids, attention_mask, ling_features)

        # Calculate loss
        loss = criterion(logits, labels)
        total_train_loss += loss.item()

        # Backward pass and optimize
        loss.backward()

        # Gradient clipping to prevent exploding gradients
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        optimizer.step()
        scheduler.step()

        # Update progress bar
        train_loop.set_description(f"Train Loss: {loss.item():.4f}")

    avg_train_loss = total_train_loss / len(train_loader)

    # --- VALIDATION ---
    model.eval()
    total_val_loss = 0
    val_preds = []
    val_true = []

    print("Running Validation...")
    with torch.no_grad():
        for batch in val_loader:
            input_ids = batch['input_ids'].to(device)
            attention_mask = batch['attention_mask'].to(device)
            ling_features = batch['ling_features'].to(device)
            labels = batch['labels'].to(device)

            logits = model(input_ids, attention_mask, ling_features)
            loss = criterion(logits, labels)
            total_val_loss += loss.item()

            # Get predictions
            preds = torch.argmax(logits, dim=1).cpu().numpy()
            val_preds.extend(preds)
            val_true.extend(labels.cpu().numpy())

    avg_val_loss = total_val_loss / len(val_loader)
    val_accuracy = accuracy_score(val_true, val_preds)

    epoch_time = time.time() - start_time
    print(f"Epoch {epoch+1} Summary: Time: {epoch_time:.2f}s | Train Loss: {avg_train_loss:.4f} | Val Loss: {avg_val_loss:.4f} | Val Accuracy: {val_accuracy*100:.2f}%")

# ==========================================
# 3. Final Preliminary Evaluation
# ==========================================
print("\n--- Final Preliminary Classification Report (Validation Set) ---")
target_names = ['Real (0)', 'Fake (1)']
print(classification_report(val_true, val_preds, target_names=target_names))

Training on device: cuda


Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

RobertaModel LOAD REPORT from: roberta-base
Key                             | Status     | 
--------------------------------+------------+-
lm_head.layer_norm.bias         | UNEXPECTED | 
lm_head.dense.bias              | UNEXPECTED | 
roberta.embeddings.position_ids | UNEXPECTED | 
lm_head.dense.weight            | UNEXPECTED | 
lm_head.layer_norm.weight       | UNEXPECTED | 
lm_head.bias                    | UNEXPECTED | 
pooler.dense.bias               | MISSING    | 
pooler.dense.weight             | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.



Starting Training Phase...

======== Epoch 1 / 3 ========


Train Loss: 0.6915: 100%|██████████| 320/320 [07:49<00:00,  1.47s/it]


Running Validation...
Epoch 1 Summary: Time: 488.27s | Train Loss: 0.6739 | Val Loss: 0.6518 | Val Accuracy: 62.07%

======== Epoch 2 / 3 ========


Train Loss: 0.6075: 100%|██████████| 320/320 [07:48<00:00,  1.46s/it]


Running Validation...
Epoch 2 Summary: Time: 487.58s | Train Loss: 0.6233 | Val Loss: 0.6350 | Val Accuracy: 63.86%

======== Epoch 3 / 3 ========


Train Loss: 0.3560: 100%|██████████| 320/320 [07:48<00:00,  1.46s/it]


Running Validation...
Epoch 3 Summary: Time: 487.63s | Train Loss: 0.5513 | Val Loss: 0.6535 | Val Accuracy: 66.98%

--- Final Preliminary Classification Report (Validation Set) ---
              precision    recall  f1-score   support

    Real (0)       0.65      0.77      0.71       668
    Fake (1)       0.69      0.56      0.62       616

    accuracy                           0.67      1284
   macro avg       0.67      0.67      0.66      1284
weighted avg       0.67      0.67      0.67      1284



### Evaluation

In [19]:
# ==========================================
# Evaluate on the LIAR Test Set
# ==========================================
import os
import pandas as pd
import torch
from torch.utils.data import DataLoader
from sklearn.metrics import classification_report, accuracy_score

print("\nPreparing LIAR Test Set...")
# 1. Load the Test set using the path we defined earlier via Kagglehub
liar_test_df = pd.read_csv(os.path.join(path, 'test.tsv'), sep='\t', names=liar_columns)

# 2. Binarize labels: 'pants-fire', 'false', 'barely-true' -> 1 (Fake)
fake_labels = ['pants-fire', 'false', 'barely-true']
liar_test_df['binary_label'] = liar_test_df['label'].apply(lambda x: 1 if x in fake_labels else 0)

# Extract texts and labels
liar_test_texts = liar_test_df['statement'].fillna("").tolist()
liar_test_labels = liar_test_df['binary_label'].tolist()

print(f"LIAR Test loaded successfully: {len(liar_test_texts)} samples.")

# 3. Create the PyTorch Dataset and DataLoader
liar_pytorch_test = FakeNewsDataset(liar_test_texts, liar_test_labels, tokenizer)
test_loader = DataLoader(liar_pytorch_test, batch_size=32, shuffle=False)

# ==========================================
# Test Evaluation Loop
# ==========================================
print("Running Evaluation on Test Set...")
model.eval()  # Set model to evaluation mode
test_preds = []
test_true = []

with torch.no_grad():
    for batch in test_loader:
        # Move inputs to the GPU
        input_ids = batch['input_ids'].to(device)
        attention_mask = batch['attention_mask'].to(device)
        ling_features = batch['ling_features'].to(device)
        labels = batch['labels'].to(device)

        # Forward pass
        logits = model(input_ids, attention_mask, ling_features)

        # Get predictions
        preds = torch.argmax(logits, dim=1).cpu().numpy()
        test_preds.extend(preds)
        test_true.extend(labels.cpu().numpy())

# Calculate final metrics
test_accuracy = accuracy_score(test_true, test_preds)
print(f"\n--- Final Test Set Accuracy: {test_accuracy*100:.2f}% ---")
print("\n--- Test Set Classification Report ---")
target_names = ['Real (0)', 'Fake (1)']
print(classification_report(test_true, test_preds, target_names=target_names))


Preparing LIAR Test Set...
LIAR Test loaded successfully: 1267 samples.
Running Evaluation on Test Set...

--- Final Test Set Accuracy: 66.30% ---

--- Test Set Classification Report ---
              precision    recall  f1-score   support

    Real (0)       0.67      0.78      0.72       714
    Fake (1)       0.64      0.52      0.57       553

    accuracy                           0.66      1267
   macro avg       0.66      0.65      0.65      1267
weighted avg       0.66      0.66      0.66      1267



In [20]:
# ==========================================
# Evaluate on the FakeNewsNet (FNN) Dataset
# ==========================================
from sklearn.model_selection import train_test_split
from torch.utils.data import DataLoader
from sklearn.metrics import classification_report, accuracy_score
import torch

print("\nPreparing FakeNewsNet Test Set...")

# 1. Split the FNN data we loaded earlier (80% train, 20% test)
# We only need the test set for this evaluation
_, fnn_test_texts, _, fnn_test_labels = train_test_split(
    fnn_texts, fnn_labels, test_size=0.2, random_state=42, stratify=fnn_labels
)

print(f"FakeNewsNet Test Set ready: {len(fnn_test_texts)} samples.")

# 2. Create the PyTorch Dataset and DataLoader
# We use max_len=512 here because FNN contains full articles, not short statements
fnn_pytorch_test = FakeNewsDataset(fnn_test_texts, fnn_test_labels, tokenizer, max_len=512)
fnn_test_loader = DataLoader(fnn_pytorch_test, batch_size=16, shuffle=False) # Smaller batch size for 512 max_len

# ==========================================
# FNN Cross-Domain Evaluation Loop
# ==========================================
print("Running Cross-Domain Evaluation on FakeNewsNet...")
model.eval()  # Ensure model is in evaluation mode
fnn_preds = []
fnn_true = []

with torch.no_grad():
    for batch in fnn_test_loader:
        # Move inputs to the GPU
        input_ids = batch['input_ids'].to(device)
        attention_mask = batch['attention_mask'].to(device)
        ling_features = batch['ling_features'].to(device)
        labels = batch['labels'].to(device)

        # Forward pass
        logits = model(input_ids, attention_mask, ling_features)

        # Get predictions
        preds = torch.argmax(logits, dim=1).cpu().numpy()
        fnn_preds.extend(preds)
        fnn_true.extend(labels.cpu().numpy())

# Calculate final metrics
fnn_test_accuracy = accuracy_score(fnn_true, fnn_preds)
print(f"\n--- FakeNewsNet Cross-Domain Test Accuracy: {fnn_test_accuracy*100:.2f}% ---")
print("\n--- FakeNewsNet Classification Report ---")
target_names = ['Real (0)', 'Fake (1)']
print(classification_report(fnn_true, fnn_preds, target_names=target_names))


Preparing FakeNewsNet Test Set...
FakeNewsNet Test Set ready: 4640 samples.
Running Cross-Domain Evaluation on FakeNewsNet...

--- FakeNewsNet Cross-Domain Test Accuracy: 26.14% ---

--- FakeNewsNet Classification Report ---
              precision    recall  f1-score   support

    Real (0)       0.82      0.02      0.04      3489
    Fake (1)       0.25      0.99      0.40      1151

    accuracy                           0.26      4640
   macro avg       0.54      0.50      0.22      4640
weighted avg       0.68      0.26      0.13      4640

